# Практика 7. Загрузка и выгрузка данных из внешних источников

На прошлой паре данные уже лежали в аккуратном CSV. В жизни так не бывает: данные приходят
из 1С в кодировке cp1251, справочники — в Excel с мусорной шапкой, клиенты — вложенным JSON
из CRM, статусы — из API, который периодически отвечает ошибкой. И всё это нужно собрать
в одну таблицу, **не потеряв и не продублировав ни одной строки**.

**Задача пары:** собрать отчёт по рейсам за апрель из четырёх источников и доказать,
что при загрузке ничего не потерялось.

| Часть | Источник | Чему учит |
|---|---|---|
| 1 | CSV из 1С | кодировки, разделители, десятичная запятая, типы |
| 2 | Excel-справочник | листы, лишние строки, `skiprows` |
| 3 | JSON из CRM | вложенные структуры, `json_normalize` |
| 4 | PostgreSQL | подключение, `read_sql`, `to_sql`, параметры запроса |
| 5 | API | пагинация, ошибки сети, повторные попытки |
| 6 | — | контроль корректности и целостности загрузки |
| 7 | — | выгрузка результата: CSV, XLSX, SQL |

> **Главная мысль пары:** загрузка данных — не «прочитать файл». Это прочитать, привести типы,
> проверить, что прочиталось всё и ровно один раз, и только потом считать. Пропущенный шаг
> проверки стоит дороже всех остальных: ошибка загрузки портит любые дальнейшие вычисления,
> и заметить её по результату почти невозможно.

---
## Шаг 0. Проект и зависимости

```
# терминал
mkdir data-sources
cd data-sources
```

`requirements.txt` — сегодня библиотек больше:

```
pandas
openpyxl
SQLAlchemy
psycopg2-binary
python-dotenv
requests
```

| Библиотека | Зачем |
|---|---|
| `openpyxl` | pandas не умеет читать `.xlsx` сам, он просит эту библиотеку |
| `SQLAlchemy` | единый способ подключаться к любой базе |
| `psycopg2-binary` | драйвер именно PostgreSQL |
| `python-dotenv` | читать пароли из файла `.env`, а не писать их в коде |
| `requests` | запросы к API по HTTP |

```
# терминал (Windows)
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
```

```
# терминал (macOS / Linux)
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

**`.gitignore` сегодня особенно важен** — добавьте в него не только окружение, но и файл с паролями:

```
.venv/
__pycache__/
.ipynb_checkpoints/
.env
```

---
## Источники данных

Запустите ячейку — она создаст три файла ровно в том виде, в каком такие выгрузки приходят.
Обратите внимание на последнюю строку вывода: это содержимое файла в байтах, **до**
расшифровки. Русский текст выглядит как `\xc5\xea\xe0...` — потому что байты сами по себе
ничего не значат, пока вы не скажете, в какой кодировке их читать.

In [ ]:
# ЯЧЕЙКА-ГЕНЕРАТОР ИСТОЧНИКОВ. Запустите её (Shift+Enter), менять ничего не нужно.
# Создаёт четыре файла — ровно в таком виде данные и приходят аналитику из разных систем.

import csv
import json
import random
from pathlib import Path

random.seed(7)
Path("data").mkdir(exist_ok=True)

routes = {
    "Москва-Казань": 820, "Москва-Санкт-Петербург": 710, "Москва-Нижний Новгород": 420,
    "Москва-Воронеж": 520, "Казань-Самара": 360, "Екатеринбург-Челябинск": 220,
    "Новосибирск-Омск": 680, "Санкт-Петербург-Петрозаводск": 430,
}
carriers = ["ТрансЛогистик", "РусКарго", "СибПуть", "ВолгаТранс", "АвтоДорСервис"]
cargo_types = ["Контейнер", "Нефтепродукты", "Зерно", "Металл", "Лес", "Стройматериалы"]
clients = ["Агрохолдинг Нива", "МеталлПром", "СтройБаза 77", "НефтеТрейд", "ЛесЭкспорт", "РитейлСеть"]

tariffs = {"Контейнер": 7.5, "Нефтепродукты": 9.2, "Зерно": 5.1,
           "Металл": 6.8, "Лес": 4.6, "Стройматериалы": 5.9}

trips = []
for i in range(1, 251):
    route, distance = random.choice(list(routes.items()))
    cargo = random.choice(cargo_types)
    weight = round(random.uniform(3, 22), 1)
    cost = distance * weight * tariffs[cargo] * random.uniform(0.95, 1.08)

    # у 5% рейсов оператор потерял разряд — стоимость в 10 раз меньше настоящей
    if random.random() < 0.05:
        cost = cost / 10

    trips.append({
        "trip_id": f"T{i:04d}",
        "date": f"{random.randint(1, 30):02d}.04.2025",          # ДД.ММ.ГГГГ — как в 1С
        "route": route,
        "carrier": random.choice(carriers),
        "cargo_type": cargo,
        # у нескольких рейсов клиент не заполнен
        "client": "" if random.random() < 0.012 else random.choice(clients),
        "weight_t": f"{weight:.1f}".replace(".", ","),           # десятичная запятая
        "distance_km": distance,
        "cost_rub": f"{cost:.2f}".replace(".", ","),
    })

# выгрузка сломалась и часть рейсов выгрузилась дважды — классика обмена между системами
trips = trips + [dict(trips[i]) for i in (5, 42, 100, 201)]

# 1) Выгрузка из 1С: кодировка cp1251, разделитель ";", десятичная запятая
with open("data/trips_1c.csv", "w", encoding="cp1251", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(trips[0].keys()), delimiter=";")
    writer.writeheader()
    writer.writerows(trips)

# 2) Справочник тарифов в Excel: две строки "шапки" сверху и два листа
from openpyxl import Workbook

wb = Workbook()
ws = wb.active
ws.title = "Тарифы"
ws.append(["Справочник тарифов на перевозку"])        # мусорная строка 1
ws.append(["Актуально на 01.04.2025"])                # мусорная строка 2
ws.append(["cargo_type", "tariff_rub_tkm"])           # настоящий заголовок
for cargo, tariff in zip(cargo_types, [7.5, 9.2, 5.1, 6.8, 4.6, 5.9]):
    ws.append([cargo, tariff])

ws2 = wb.create_sheet("Перевозчики")
ws2.append(["carrier", "rating"])
for c, r in zip(carriers, [4.6, 4.1, 3.8, 4.4, 3.5]):
    ws2.append([c, r])

wb.save("data/tariffs.xlsx")

# 3) Клиенты: вложенный JSON, как отдаёт CRM
clients_json = []
for name in clients:
    clients_json.append({
        "client": name,
        "inn": str(random.randint(5000000000, 7999999999)),
        "contract": {"number": f"D-{random.randint(100, 999)}", "signed": "2024-11-15"},
        "contacts": [
            {"type": "email", "value": f"info@{random.randint(10, 99)}.ru"},
            {"type": "phone", "value": f"+7999{random.randint(1000000, 9999999)}"},
        ],
    })

Path("data/clients.json").write_text(
    json.dumps(clients_json, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Готово. Создано:")
print("  data/trips_1c.csv  — выгрузка из 1С: cp1251, разделитель ';', запятая в числах")
print("  data/tariffs.xlsx  — Excel: 2 мусорные строки сверху, листы 'Тарифы' и 'Перевозчики'")
print("  data/clients.json  — CRM: вложенная структура")
print()
print("Строк в выгрузке 1С:", len(trips), "(из них 4 — повторы)")
print()
print("Первые 300 байт trips_1c.csv, прочитанные КАК ЕСТЬ (без кодировки):")
print(open("data/trips_1c.csv", "rb").read(300))

---
# Часть 1. CSV: кодировки, разделители, типы

**Кодировка** — таблица соответствия «байт → символ». Текстовый файл не хранит в себе
информацию о своей кодировке, поэтому программа должна её знать или угадать.
Две кодировки, которые вы встретите в работе:

- **UTF-8** — современный стандарт, по умолчанию во всём;
- **cp1251** (она же Windows-1251) — русская кодировка из 90-х, до сих пор живёт в выгрузках
  из 1С, старых учётных систем и Excel под Windows.

Если прочитать cp1251 как UTF-8, программа либо упадёт с `UnicodeDecodeError`, либо выдаст
«кракозябры» — `Ðœ Ð¾ÑÐºÐ²Ð°` вместо «Москва».

In [ ]:
import pandas as pd

# Попытка 1: читаем как обычно — pandas предполагает UTF-8
try:
    df = pd.read_csv("data/trips_1c.csv")
    print(df.head())
except UnicodeDecodeError as e:
    print("UnicodeDecodeError — байты не расшифровались как UTF-8:")
    print("  ", e)

In [ ]:
# Попытка 2: кодировку указали, а разделитель — нет.
# pandas по умолчанию ждёт запятую, а в файле точка с запятой
df = pd.read_csv("data/trips_1c.csv", encoding="cp1251")
print("столбцов получилось:", df.shape[1])
print("названия столбцов:", list(df.columns))
print()
print(df.head(2))

Один столбец вместо девяти — характерный признак неверного разделителя. Таблица
«прочиталась», ошибки не было, но дальше с ней работать нельзя. Это важный тип ошибки:
**она не падает, она тихо даёт неправильный результат.**

Основные параметры `read_csv`, которые решают такие проблемы:

| Параметр | Зачем |
|---|---|
| `encoding="cp1251"` | кодировка файла |
| `sep=";"` | разделитель столбцов |
| `decimal=","` | десятичная запятая вместо точки |
| `thousands=" "` | разделитель разрядов (`1 250,50`) |
| `parse_dates=["date"], dayfirst=True` | разобрать даты формата `ДД.ММ.ГГГГ` |
| `dtype={"trip_id": str}` | принудительный тип столбца |
| `na_values=["н/д", "-"]` | что считать пропуском |
| `usecols=[...]`, `nrows=100` | прочитать только часть — удобно для разведки |

In [ ]:
# Попытка 3: всё по-честному
trips = pd.read_csv(
    "data/trips_1c.csv",
    encoding="cp1251",
    sep=";",
    decimal=",",
    parse_dates=["date"],
    dayfirst=True,
)

print("размер:", trips.shape)
print()
print(trips.dtypes)
print()
trips.head()

### Задание 1

1. Прочитайте `data/trips_1c.csv` **без** параметра `decimal=","` и посмотрите на `dtypes`.
   Какой тип получили столбцы `weight_t` и `cost_rub`? Почему так?
2. Попробуйте посчитать `.sum()` по столбцу `cost_rub` из этой «неправильной» таблицы.
   Что получилось вместо суммы и почему?
3. Прочитайте файл правильно и убедитесь, что оба столбца стали числовыми.
4. Посчитайте суммарную стоимость всех рейсов — теперь по-настоящему.

In [ ]:
# 1) читаем без decimal
bad = ...
print(bad[["weight_t", "cost_rub"]].dtypes)

# 2) что даст sum() на таком столбце?
print(...)

# 3) читаем правильно
good = ...
print(good[["weight_t", "cost_rub"]].dtypes)

# 4) суммарная стоимость
print("сумма:", ...)

---
# Часть 2. Excel

`.xlsx` — не текстовый файл, а архив с XML внутри, поэтому кодировка здесь не проблема.
Зато есть свои: несколько листов, объединённые ячейки, «красивые» шапки из нескольких строк,
числа, сохранённые как текст.

| Параметр `read_excel` | Зачем |
|---|---|
| `sheet_name="Тарифы"` | какой лист читать (можно номер; `None` — все сразу) |
| `skiprows=2` | пропустить первые строки до настоящего заголовка |
| `header=None` | в файле вообще нет заголовка |
| `usecols="A:C"` | только нужные столбцы |

Полезный приём для разведки — `pd.ExcelFile`: позволяет сначала посмотреть список листов,
не читая данные.

In [ ]:
# Сначала смотрим, что вообще внутри файла
xls = pd.ExcelFile("data/tariffs.xlsx")
print("листы:", xls.sheet_names)
print()

# Читаем как есть — видим мусорную шапку
raw = pd.read_excel("data/tariffs.xlsx", sheet_name="Тарифы")
print("как есть:")
print(raw.head())

In [ ]:
# Пропускаем две лишние строки — заголовок встаёт на место
tariffs = pd.read_excel("data/tariffs.xlsx", sheet_name="Тарифы", skiprows=2)
print(tariffs)
print()
print(tariffs.dtypes)

### Задание 2

1. Прочитайте лист `Перевозчики` из того же файла в переменную `carriers_df`.
   Обратите внимание: на этом листе мусорной шапки нет — `skiprows` не нужен.
2. Превратите таблицу тарифов в словарь `{тип груза: тариф}`. Подсказка: метод
   `.set_index(...)[...]` и затем `.to_dict()`.
3. Подставьте тариф в таблицу рейсов новым столбцом `tariff` (через `.map()`, как на прошлой паре).
4. **Проверьте подстановку:** сколько получилось пропусков в столбце `tariff`?
   Если больше нуля — значит в рейсах есть тип груза, которого нет в справочнике.
   Это типичная причина «потери» данных при соединении источников.

In [ ]:
# 1)
carriers_df = ...
print(carriers_df)

# 2)
tariff_map = ...
print(tariff_map)

# 3)
trips["tariff"] = ...

# 4)
print("пропусков в tariff:", ...)

---
# Часть 3. JSON

JSON хранит не таблицу, а **дерево**: словари внутри словарей, списки внутри словарей.
Поэтому прямого превращения в таблицу нет — сначала нужно решить, что считать строкой.

Три инструмента:

| Инструмент | Когда |
|---|---|
| `json.load(открытый_файл)` | получить обычный Python-объект и посмотреть на структуру |
| `pd.read_json("file.json")` | данные уже плоские — список одинаковых словарей |
| `pd.json_normalize(объект)` | есть вложенность: разворачивает `contract.number` в столбец |

Правило: сначала посмотрите на структуру через `json.load`, потом выбирайте инструмент.

In [ ]:
import json

with open("data/clients.json", encoding="utf-8") as f:
    raw_clients = json.load(f)

print("тип:", type(raw_clients), "| записей:", len(raw_clients))
print()
print("первая запись:")
print(json.dumps(raw_clients[0], ensure_ascii=False, indent=2))

In [ ]:
# json_normalize разворачивает вложенные словари: contract.number, contract.signed
clients = pd.json_normalize(raw_clients)
print(clients.columns.tolist())
print()
print(clients[["client", "inn", "contract.number"]])
print()
print("а contacts остался списком:", type(clients["contacts"].iloc[0]))

In [ ]:
# Если нужна таблица по вложенному списку — указываем record_path.
# Каждый контакт станет отдельной строкой, а поля из meta продублируются.
contacts = pd.json_normalize(
    raw_clients,
    record_path="contacts",
    meta=["client", "inn"],
)
print(contacts.head(6))
print()
print("строк:", len(contacts), "— по два контакта на каждого из", len(raw_clients), "клиентов")

### Задание 3

1. Создайте словарь `inn_map` вида `{название клиента: ИНН}` и подставьте ИНН
   в таблицу рейсов столбцом `inn`.
2. Из таблицы `contacts` получите только электронные адреса (`type == "email"`)
   и сделайте словарь `{клиент: email}`.
3. Сколько рейсов осталось без ИНН? Посмотрите на эти строки — в чём причина?
   (Подсказка: посмотрите на столбец `client` у этих рейсов.)

In [ ]:
# 1)
inn_map = ...
trips["inn"] = ...

# 2)
emails = ...
print(emails)

# 3)
print("рейсов без ИНН:", ...)
...

---
# Часть 4. PostgreSQL

## 4.1 Пароль не должен попасть в репозиторий

Самая частая ошибка на этой теме — написать пароль прямо в ноутбуке, а потом запушить его
на GitHub. Пароль, попавший в историю git, считается скомпрометированным навсегда:
удалить его из истории сложно, а публичные репозитории сканируют боты.

Два безопасных способа:

**Способ 1 — спросить пароль при запуске** (используем сегодня, ничего настраивать не нужно):

```python
import getpass
password = getpass.getpass("Пароль: ")      # ввод не отображается и не сохраняется в ноутбуке
```

**Способ 2 — файл `.env`** (так делают в реальных проектах). Создайте рядом с ноутбуком файл
`.env` **и добавьте его в `.gitignore`**:

```
PG_HOST=localhost
PG_PORT=5432
PG_DB=students
PG_USER=student
PG_PASSWORD=ваш_пароль
```

Параметры подключения к учебной базе вам даст преподаватель.

In [ ]:
import getpass
import os
from urllib.parse import quote_plus

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()          # подхватит .env, если он есть; если нет — просто ничего не сделает

# Если доступа к PostgreSQL сейчас нет, поставьте True — всё задание будет работать
# на локальном файле SQLite. Код обращения к базе при этом не меняется ни на строку.
USE_SQLITE = False

# Ваша фамилия латиницей: у каждого студента своя таблица, чтобы не мешать друг другу
STUDENT_ID = "ivanov"
TABLE_NAME = f"trips_{STUDENT_ID}"

if USE_SQLITE:
    engine = create_engine("sqlite:///data/local.db")
    print("Работаем с SQLite:", "data/local.db")
else:
    PG_HOST = os.getenv("PG_HOST", "localhost")
    PG_PORT = os.getenv("PG_PORT", "5432")
    PG_DB = os.getenv("PG_DB", "students")
    PG_USER = os.getenv("PG_USER", "student")
    PG_PASSWORD = os.getenv("PG_PASSWORD") or getpass.getpass("Пароль PostgreSQL: ")

    # quote_plus нужен, если в пароле есть @ : / — иначе строка подключения разберётся неверно
    url = (f"postgresql+psycopg2://{PG_USER}:{quote_plus(PG_PASSWORD)}"
           f"@{PG_HOST}:{PG_PORT}/{PG_DB}")
    engine = create_engine(url)
    print("Подключаемся к PostgreSQL:", f"{PG_HOST}:{PG_PORT}/{PG_DB}")

print("Таблица для работы:", TABLE_NAME)

In [ ]:
# Проверка связи. Если здесь ошибка — дальше идти бессмысленно, разберитесь с подключением.
with engine.connect() as conn:
    print("связь есть, ответ сервера:", conn.execute(text("SELECT 1")).scalar())
    if not USE_SQLITE:
        print(conn.execute(text("SELECT version()")).scalar())

## 4.2 Запись и чтение

| Операция | Код |
|---|---|
| записать таблицу | `df.to_sql(TABLE_NAME, engine, if_exists="replace", index=False)` |
| прочитать запросом | `pd.read_sql(text("SELECT ..."), engine)` |
| прочитать таблицу целиком | `pd.read_sql_table(TABLE_NAME, engine)` |

Параметр `if_exists` решает, что делать, если таблица уже есть:
`"fail"` — ошибка (по умолчанию), `"replace"` — удалить и создать заново,
`"append"` — добавить строки в конец.

> `index=False` почти всегда нужен: иначе в базу уедет технический индекс pandas
> отдельным столбцом, и при повторной загрузке таблица будет расти лишними полями.

### Задание 4

1. Запишите таблицу рейсов `trips` в свою таблицу в базе (`if_exists="replace"`, без индекса).
2. Прочитайте её обратно в переменную `from_db` и сравните с исходной: число строк
   и сумму `cost_rub`. Совпало?
3. Выполните SQL-запрос, который посчитает по каждому перевозчику число рейсов и суммарную
   стоимость — **только по рейсам длиннее 500 км**. Отсортируйте по сумме, по убыванию.
4. Посчитайте то же самое средствами pandas (фильтр + `groupby`) и сравните результаты.
   Два пути к одному числу — это и есть проверка, что вы ничего не напутали.

In [ ]:
# 1)
...

# 2)
from_db = ...
print("строк в pandas:", len(trips), "| строк из базы:", len(from_db))
print("сумма в pandas:", ..., "| сумма из базы:", ...)

# 3)
query = text("""
    SELECT carrier, COUNT(*) AS trips_count, SUM(cost_rub) AS revenue
    FROM ...
    WHERE ...
    GROUP BY carrier
    ORDER BY revenue DESC
""")
sql_result = ...
print(sql_result)

# 4)
pandas_result = ...
print(pandas_result)

## 4.3 Параметры в запросе, а не склейка строк

Подставлять значения в SQL склейкой строк нельзя — это и ошибки с кавычками, и прямая дыра
в безопасности (**SQL-инъекция**): текст из переменной становится частью команды.

```python
# ТАК НЕЛЬЗЯ
carrier = "РусКарго"
pd.read_sql(f"SELECT * FROM {TABLE_NAME} WHERE carrier = '{carrier}'", engine)
```

Если в переменную попадёт `'; DROP TABLE trips; --`, вы сами выполните удаление таблицы.
Правильно — передавать значения **параметрами**: база получает запрос и данные отдельно
и никогда не принимает данные за команду.

Важное исключение: параметром можно передать **значение**, но не имя таблицы или столбца.
Имя таблицы приходится вставлять в текст запроса — поэтому оно должно быть вашим собственным,
а не пришедшим извне.

> Ячейка ниже обращается к вашей таблице в базе, поэтому запускайте её **после** задания 4:
> таблица должна быть уже создана. Если увидите ошибку вида `no such table` или
> `relation does not exist` — вернитесь к пункту 1 задания 4.

In [ ]:
query = text(f"""
    SELECT carrier, cargo_type, cost_rub
    FROM {TABLE_NAME}
    WHERE carrier = :carrier AND cost_rub > :min_cost
    ORDER BY cost_rub DESC
""")

result = pd.read_sql(query, engine, params={"carrier": "РусКарго", "min_cost": 50000})
print("найдено строк:", len(result))
result.head()

---
# Часть 5. API

API отдаёт данные по сети, и это меняет правила: ответ может не прийти, прийти не полностью
или прийти с ошибкой. Поэтому загрузка из API — всегда цикл с проверками.

**Как выглядит настоящий запрос:**

```python
import requests

response = requests.get(
    "https://api.example.com/trips",
    params={"page": 1, "per_page": 50},
    timeout=10,                      # не ждать вечно
)
response.raise_for_status()          # ошибка, если код ответа не 2xx
data = response.json()               # разобрать JSON в Python-объект
```

Что обязательно проверять: **код ответа** (`response.status_code`, 200 — успех,
404 — нет такого адреса, 500 — сервер сломался), **таймаут** и **пагинация** — API почти
никогда не отдаёт все данные сразу, он выдаёт их страницами.

Чтобы пара не зависела от интернета в аудитории, API мы подменяем функцией. Она ведёт себя
как настоящая: отдаёт данные страницами и **каждый третий запрос падает с ошибкой**.

In [ ]:
# Имитатор внешнего API. В аудитории интернет может не работать, поэтому API мы
# подменяем функцией: она ведёт себя так же, как настоящий — отдаёт JSON страницами,
# иногда отвечает ошибкой и не присылает всё сразу.

import random

_statuses = ["delivered", "in_transit", "delayed", "cancelled"]
random.seed(77)
_all_records = [
    {"trip_id": f"T{i:04d}", "status": random.choices(_statuses, weights=[70, 15, 10, 5])[0]}
    for i in range(1, 251)
]
_fail_counter = {"n": 0}


def api_get_statuses(page=1, per_page=50):
    """Отдаёт одну страницу статусов рейсов.

    Возвращает словарь: {"page": ..., "per_page": ..., "total": ..., "items": [...]}
    Каждый третий запрос "падает" с ошибкой 503 — как настоящий сервис под нагрузкой.
    """
    _fail_counter["n"] += 1
    if _fail_counter["n"] % 3 == 0:
        raise ConnectionError("503 Service Unavailable")

    start = (page - 1) * per_page
    items = _all_records[start:start + per_page]
    return {"page": page, "per_page": per_page, "total": len(_all_records), "items": items}


print("Функция api_get_statuses(page, per_page) готова.")
print("Проверка:", api_get_statuses(page=1, per_page=3))

### Задание 5

Напишите функцию `load_all_statuses()`, которая соберёт **все** статусы рейсов:

1. запрашивает страницы по 50 записей, начиная с первой;
2. при ошибке `ConnectionError` — повторяет попытку (не больше 3 раз на страницу),
   и только потом сдаётся;
3. останавливается, когда собрано `total` записей или страница вернулась пустой;
4. возвращает `DataFrame` со столбцами `trip_id` и `status`.

Затем проверьте результат: сколько записей собрано, совпадает ли с `total`,
нет ли дубликатов `trip_id`.

> Бесконечный `while True` без ограничения числа попыток — самая частая ошибка в таком коде.
> Если сервис лежит, ваш скрипт должен закончиться с понятной ошибкой, а не висеть вечно.

In [ ]:
def load_all_statuses(per_page=50, max_retries=3):
    records = []
    page = 1
    total = None

    while True:
        # 1) попытаться получить страницу; при ConnectionError повторить, но не больше max_retries раз
        # 2) запомнить total из ответа
        # 3) добавить полученные items в records
        # 4) если собрали total записей или страница вернулась пустой — выйти из цикла
        # 5) иначе перейти к следующей странице

        break   # временная заглушка, чтобы цикл не стал бесконечным: уберите её, когда напишете шаги выше

    return pd.DataFrame(records)


statuses = load_all_statuses()
print("собрано записей:", len(statuses))
print("дубликатов trip_id:", ...)
print(statuses.head())

---
# Часть 6. Контроль корректности и целостности загрузки

Самая важная часть пары. Данные загружены — но загружены ли **правильно**?
Ошибки загрузки коварны тем, что не видны в результате: отчёт посчитается, графики
нарисуются, и никто не заметит, что четыре рейса учтены дважды.

**Проверочный список после любой загрузки:**

| Что проверяем | Чем |
|---|---|
| число строк совпадает с ожидаемым | `len(df)` против известного числа |
| ключ уникален | `df["trip_id"].duplicated().sum()` |
| нет пропусков в обязательных полях | `df[cols].isna().sum()` |
| типы столбцов те, что нужны | `df.dtypes` |
| значения в допустимых границах | `df["cost_rub"].min()`, `.max()` |
| даты попадают в ожидаемый период | `df["date"].min()`, `.max()` |
| контрольная сумма сходится | `df["cost_rub"].sum()` до и после обработки |
| запись-чтение не искажает данные | записать, прочитать, сравнить |

In [ ]:
def check_load(df, key=None, required=(), expected_rows=None, checksum_col=None):
    """Печатает отчёт о качестве загрузки. OK — всё хорошо, ВНИМАНИЕ — надо разбираться."""
    print("ПРОВЕРКА ЗАГРУЗКИ")
    print("-" * 46)
    print(f"строк: {len(df)}", end="")
    if expected_rows is not None:
        print(" | ожидали:", expected_rows, "->", "OK" if len(df) == expected_rows else "ВНИМАНИЕ")
    else:
        print()

    if key:
        dups = df[key].duplicated().sum()
        print(f"дубликаты {key}: {dups} ->", "OK" if dups == 0 else "ВНИМАНИЕ")

    for col in required:
        n = df[col].isna().sum()
        print(f"пропуски в {col}: {n} ->", "OK" if n == 0 else "ВНИМАНИЕ")

    if checksum_col:
        print(f"контрольная сумма {checksum_col}: {df[checksum_col].sum():,.2f}".replace(",", " "))
    print("-" * 46)


check_load(trips, key="trip_id", required=["date", "cost_rub"],
           expected_rows=250, checksum_col="cost_rub")

### Задание 6

Разберитесь с тем, что показала проверка, и приведите данные в порядок.

1. Сколько в выгрузке дубликатов `trip_id`? Удалите их (`drop_duplicates(subset=...)`)
   и **запишите контрольную сумму `cost_rub` до и после**. На сколько рублей отличается?
2. Какие рейсы остались без клиента? Что с ними делать — решите и обоснуйте.
3. **Проверка на осмысленность значений.** Посчитайте ожидаемую стоимость каждого рейса
   как `weight_t * distance_km * tariff` и сравните с фактической `cost_rub`.
   Сколько рейсов отличаются от расчётной более чем в два раза? Посмотрите на эти строки:
   какую ошибку допустил оператор при вводе?
4. Проверьте, что все даты попадают в апрель 2025 года.
5. **Round-trip тест.** Запишите очищенную таблицу в `data/check.csv`, прочитайте обратно
   и сравните число строк и сумму `cost_rub` с исходной. Совпало?

In [ ]:
# 1) дубликаты
print("сумма до дедупликации:", ...)
clean = ...
print("сумма после:", ...)
print("разница:", ...)

# 2) рейсы без клиента
...

# 3) аномалии стоимости
clean["expected_cost"] = ...
clean["ratio"] = ...
anomalies = ...
print("аномалий:", len(anomalies))
...

# 4) период
print("даты:", ..., "-", ...)

# 5) round-trip
...

---
# Часть 7. Выгрузка результата

Результат нужно отдать в том виде, в котором его примет получатель: коллеге — Excel,
в хранилище — базу, другой программе — CSV или JSON.

| Куда | Как | Что не забыть |
|---|---|---|
| CSV | `df.to_csv("out.csv", index=False, encoding="utf-8-sig")` | `index=False`; `utf-8-sig` — чтобы Excel не показал кракозябры |
| Excel, один лист | `df.to_excel("out.xlsx", index=False)` | нужен `openpyxl` |
| Excel, несколько листов | `pd.ExcelWriter` | каждый лист своим `to_excel` |
| JSON | `df.to_json(..., orient="records", force_ascii=False)` | `force_ascii=False` — чтобы русский остался русским |
| база | `df.to_sql(name, engine, if_exists="replace", index=False)` | `index=False` |

### Задание 7

Выгрузите результат тремя способами:

1. **CSV:** очищенную таблицу в `data/trips_clean.csv` — без индекса, в `utf-8-sig`.
   Откройте файл в Excel и убедитесь, что русский текст читается.
2. **Excel с двумя листами:** лист `Рейсы` — очищенные данные, лист `Сводка` — выручка
   и число рейсов по перевозчикам. Используйте `pd.ExcelWriter`.
3. **База:** запишите очищенную таблицу в `{TABLE_NAME}_clean` и проверьте запросом
   `SELECT COUNT(*)`, что строк столько, сколько ожидаете.

In [ ]:
# 1) CSV
...

# 2) Excel с двумя листами
summary = ...

with pd.ExcelWriter("data/report.xlsx") as writer:
    ...

# 3) база
...

---
## Отправляем работу

```
# терминал
git status          # убедитесь, что .env в списке НЕ значится
git add .
git commit -m "Практика 7: загрузка из CSV, Excel, JSON, SQL и API с контролем целостности"
git push
```

> Перед пушем отдельно проверьте вывод `git status`. Если видите там `.env` — остановитесь
> и добавьте его в `.gitignore`. Пароль, попавший в историю git, оттуда уже не удаляется.

## Чек-лист сдачи

- [ ] Задание 1: объяснено, почему без `decimal=","` столбцы стали текстовыми, файл прочитан верно
- [ ] Задание 2: оба листа Excel прочитаны, тариф подставлен, подстановка проверена на пропуски
- [ ] Задание 3: ИНН подставлен из JSON, адреса получены, причина пропусков объяснена
- [ ] Задание 4: таблица записана в базу и прочитана обратно, SQL-агрегация совпала с pandas
- [ ] Задание 5: все страницы API собраны, повторные попытки работают, дубликатов нет
- [ ] Задание 6: дубликаты удалены, расхождение контрольной суммы посчитано, аномалии найдены
- [ ] Задание 7: три выгрузки — CSV, Excel с двумя листами, таблица в базе
- [ ] `.env` не попал в репозиторий

## Что унести с пары

Ошибки загрузки — самый дорогой вид ошибок в аналитике, потому что они **не выглядят как
ошибки**. Неверный разделитель даёт один столбец вместо девяти, и это видно сразу.
А четыре продублированные строки дают завышенную выручку, которая выглядит совершенно
нормально — и попадает в отчёт для руководства.

Отсюда рабочее правило: **после каждой загрузки — проверка**. Число строк, уникальность ключа,
контрольная сумма. Три строки кода, которые отделяют аналитика от генератора красивых чисел.